In [ ]:
import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer)

# 필요한 라이브러리를 설치합니다.
!pip install transformers datasets accelerate evaluate sacrebleu sentencepiece -q

이제 Hugging Face에서 `shihyunlim/aihub-ko-en-everyday-expression` 데이터셋을 로드합니다. 데이터셋은 한국어 문장(`ko`)과 그에 해당하는 영어 번역(`en`)으로 구성되어 있습니다.

In [ ]:
dataset_id = "shihyunlim/aihub-ko-en-everyday-expression"
raw_datasets = load_dataset(dataset_id)

print(raw_datasets)

다음으로, 토크나이저를 로드하고 데이터셋의 `train` 분할에 있는 샘플의 토큰화를 시도하여 토크나이저가 어떻게 작동하는지 확인합니다. 이 경우에는 `Helsinki-NLP/opus-mt-ko-en` 모델을 위한 `AutoTokenizer`를 사용합니다.

In [ ]:
model_checkpoint = "Helsinki-NLP/opus-mt-ko-en"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

sample_text = raw_datasets["train"][0]["translation"]["ko"]
tokenized_sample = tokenizer(sample_text)

print(f"Original text: {sample_text}")
print(f"Tokenized text: {tokenizer.convert_ids_to_tokens(tokenized_sample['input_ids'])}")
print(f"Input IDs: {tokenized_sample['input_ids']}")

이제 데이터셋의 모든 샘플에 토크나이저를 적용하는 전처리 함수를 정의하겠습니다. `truncation=True`를 설정하여 모델의 최대 시퀀스 길이로 자릅니다.

In [ ]:
max_input_length = 128
max_target_length = 128

def preprocess_function(examples):
    inputs = [ex["ko"] for ex in examples["translation"]]
    targets = [ex["en"] for ex in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)
    labels = tokenizer(targets, max_length=max_target_length, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = raw_datasets.map(preprocess_function, batched=True)

print(tokenized_datasets)

다음으로 `Helsinki-NLP/opus-mt-ko-en` 체크포인트에서 `AutoModelForSeq2SeqLM`을 로드합니다. 또한 효율적인 배치를 위해 `DataCollatorForSeq2Seq`를 정의합니다.

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

모델의 성능을 평가하기 위해 BLEU 점수를 계산하는 `compute_metrics` 함수를 정의합니다. 또한 `Seq2SeqTrainingArguments`를 설정하여 학습 파라미터를 정의합니다. 요청에 따라 1 epoch만 학습합니다.

In [ ]:
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["bleu"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    return result

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1, # 1 epoch만 학습
    predict_with_generate=True,
    fp16=torch.cuda.is_available(), # GPU를 사용하는 경우 fp16 활성화
    push_to_hub=False,
    report_to=[] # wandb와 같은 보고 도구를 비활성화
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()


학습이 완료된 후, 모델을 평가하고 BLEU 점수를 확인할 수 있습니다.

In [ ]:
eval_results = trainer.evaluate()
print(eval_results)

마지막으로, 번역을 테스트하기 위해 샘플 문장을 가져와 모델을 사용하여 번역합니다.

In [ ]:
input_text = "안녕하세요, 오늘 기분 어떠세요?"
inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=max_target_length)
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Korean: {input_text}")
print(f"English translation: {decoded_output}")

### 하이퍼파라미터 탐색

하이퍼파라미터 탐색의 경우, `Seq2SeqTrainingArguments`의 `learning_rate`, `per_device_train_batch_size`, `weight_decay`와 같은 인수를 조정할 수 있습니다. 예를 들어, `num_train_epochs`를 1보다 큰 값으로 늘리거나 다른 최적화 도구(예: AdamW)를 사용하여 학습 속도를 변경할 수 있습니다.

또한 Hugging Face `Trainer`는 Optuna나 Ray Tune과 같은 라이브러리를 사용하여 하이퍼파라미터 탐색을 자동화하는 기능을 지원합니다. `Trainer` 객체를 생성할 때 `model_init` 함수와 `hp_space` 인수를 정의하여 사용할 수 있습니다. 아래는 간략한 예시입니다.

```python
def model_init():
    return AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

trainer = Seq2SeqTrainer(
    model_init=model_init,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 학습할 하이퍼파라미터 공간 정의
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical("per_device_train_batch_size", [8, 16, 32]),
    }

best_run = trainer.hyperparameter_search(
    direction="minimize", # 또는 "maximize" (예: BLEU 점수)
    backend="optuna",
    hp_space=hp_space,
    n_trials=10, # 시도할 하이퍼파라미터 조합 수
)

print(best_run)
```

이 예시는 Optuna를 백엔드로 사용하여 학습 속도와 배치 크기를 탐색합니다. `n_trials`를 늘려 더 많은 조합을 시도할 수 있습니다. 실제 프로덕션 환경에서는 이와 같은 자동화된 하이퍼파라미터 탐색이 모델 성능을 최적화하는 데 도움이 될 수 있습니다.